# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print summary information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'unknown')}")

## 2. Data Overview
Let's review the available record sets, their fields, and corresponding `@id` properties.

The `@id` uniquely identifies each entity within the Croissant schema. All further references to record sets, fields, and columns will use these IDs.

In [ ]:
# Inspect and list all available record sets
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
all_record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, type: {f.data_type})")
    print()
    all_record_set_ids.append(rs.id)
# Preview records from first record set (if available)
if record_sets:
    first_id = record_sets[0].id
    print(f"First 1-2 records from record set '{record_sets[0].name}':")
    for idx, rec in enumerate(dataset.records(record_set=first_id)):
        print(rec)
        if idx > 0: break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the previous overview.

In [ ]:
# Load all record set data into DataFrames, key by @id
dataframes = {}
for rs_id in all_record_set_ids:
    # Use list() to fully materialize records
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
print(f"Loaded DataFrames: {list(dataframes.keys())}")
# Display columns and head for the first record set
if all_record_set_ids:
    main_rs_id = all_record_set_ids[0]
    df = dataframes[main_rs_id]
    print(f"Columns in DataFrame (from RecordSet '@id': {main_rs_id}):\n", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's explore a numeric field, filter and normalize it, and group by a categorical field. All field references are by their `@id` as required by the Croissant schema.

We will select the first numeric field and first string/categorical field from main record set for demonstration.

In [ ]:
# Find numeric and group fields by inspection of fields' Croissant @id
# We'll choose common proteomics fields: MW (molecular weight), Coverage, or Count fields if present
rs = [r for r in dataset.record_sets if r.id == main_rs_id][0]
numeric_field_candidates = [f for f in rs.fields if f.data_type in ['Float', 'Integer', 'Number']]
group_field_candidates = [f for f in rs.fields if f.data_type in ['String', 'Text']]

# Fallback defaults if automatic guess fails
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0].id
else:
    numeric_field = df.select_dtypes('number').columns[0]  # try to pick the first numeric

if group_field_candidates:
    group_field = group_field_candidates[0].id
else:
    group_field = df.select_dtypes('object').columns[0]  # fallback to first text string

print(f"Using numeric field '@id': {numeric_field}")
print(f"Using group field '@id': {group_field}\n")

# Filter (e.g., remove molecular weight < 20000 if MW present; else coverage < 10)
if numeric_field in df.columns:
    # Set a sample threshold (10 or 20000)
    threshold = 10
    if 'weight' in numeric_field.lower() or 'mw' in numeric_field.lower():
        threshold = 20000
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}, count: {len(filtered_df)}")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group (e.g., by protein name)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"Top 5 groups by mean {numeric_field}:")
        print(grouped_df.head())
else:
    print("Field for numeric analysis not found.")

## 5. Visualization
Here we'll visualize the distribution of the numeric field and the group-wise mean using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'grouped_df' in locals() and len(grouped_df) > 1:
        plt.figure(figsize=(10,4))
        # Show top 20 by mean, if there are many categories
        subset = grouped_df.head(20)
        sns.barplot(y=group_field, x=numeric_field, data=subset)
        plt.title(f"Group mean of {numeric_field} by {group_field} (top 20)")
        plt.tight_layout()
        plt.show()
else:
    print("Visualization fields not found in DataFrame.")

## 6. Conclusion
- We loaded the FAIR^2 protein dataset and inspected available record sets and fields by their Croissant `@id`.
- We demonstrated extraction, normalization, group aggregation, and simple visualizations.
- For full reference, always use the `@id` provided by the Croissant metadata when selecting fields or entities in the dataset.

**Further steps:** Explore other record sets or fields by repeating the above steps with different `@id`s as relevant to your analysis.
